# The `static` Keyword

The `static` keyword means different things in different contexts. In C you know it for file-scope variables. In C++ classes, it has a different, powerful meaning: it makes a member **belong to the class itself** rather than to any individual object.

This notebook covers:
1. `static` in C (review).
2. Static data members.
3. Static member functions.
4. The Singleton pattern.
5. `const static` members.

## `static` in C (Review)

You already know these two uses from C:

### 1. Static local variable — persists between calls

```c
void counter() {
    static int count = 0;   // initialised once, then persists
    count++;
    printf("%d\n", count);
}
```

### 2. Static global variable — file scope (internal linkage)

```c
static int privateToThisFile = 0;   // not visible to other .c files
```

In [ ]:
#include <iostream>

// Static local: persists between calls
void callCounter() {
    static int count = 0;
    count++;
    std::cout << "callCounter called " << count << " time(s)" << std::endl;
}

callCounter();   // 1
callCounter();   // 2
callCounter();   // 3

## `static` in C++ (Class Context)

Inside a class, `static` has a **completely different meaning**: the member belongs to the **class**, not to any specific instance.

```
Without static:                 With static:

obj1: [ x=10, name="Alice" ]    obj1: [ x=10, name="Alice" ]
obj2: [ x=20, name="Bob"   ]    obj2: [ x=20, name="Bob"   ]
                                Class: [ instanceCount=2    ]  <-- shared
```

All instances share the same static member. Changing it from one instance changes it for all.

## Static Data Members

Two steps are required:
1. **Declare** inside the class: `static int instanceCount;`
2. **Define** outside the class (allocates storage): `int MyClass::instanceCount = 0;`

In a real project the definition goes in the `.cpp` file. In this notebook we can write it right after the class definition.

In [ ]:
#include <iostream>

class Counter {
public:
    Counter() {
        instanceCount++;
        std::cout << "[Counter] created. Total: " << instanceCount << std::endl;
    }

    ~Counter() {
        instanceCount--;
        std::cout << "[Counter] destroyed. Total: " << instanceCount << std::endl;
    }

    static int instanceCount;   // declaration
};

// Definition (outside the class) — required in C++98
int Counter::instanceCount = 0;

std::cout << "Before any objects: " << Counter::instanceCount << std::endl;

{
    Counter c1;
    Counter c2;
    Counter c3;
    std::cout << "Inside block: " << Counter::instanceCount << std::endl;
}   // c1, c2, c3 destroyed here

std::cout << "After block: " << Counter::instanceCount << std::endl;

**Exercise 1:** Create a class `IdGenerator` with a private `static int nextId` counter. Each time an object is constructed it receives a unique sequential ID (1, 2, 3 ...) stored in a public `int id` member. Create four `IdGenerator` objects and print each one's ID.

In [ ]:
// Your code here

## Static Member Functions

A static member function:
- Belongs to the class, not to an instance.
- Can be called **without creating an object**: `MyClass::myStaticFunc()`.
- Cannot access non-static members (there is no `this` pointer).
- Can access other static members.

```cpp
class MyClass {
public:
    static void printCount();
    static int instanceCount;
};

MyClass::printCount();   // call without any object
```

In [ ]:
#include <iostream>

class Robot {
public:
    Robot(const char *name) : _name(name) {
        _count++;
    }

    ~Robot() {
        _count--;
    }

    // Static member function
    static void printCount() {
        std::cout << "Active robots: " << _count << std::endl;
        // Cannot use _name here -- no 'this'
    }

    void introduce() const {
        std::cout << "I am " << _name << std::endl;
    }

private:
    const char *_name;
    static int  _count;
};

int Robot::_count = 0;

// Call static method without any object
Robot::printCount();

Robot r1("R2D2");
Robot r2("C3PO");
Robot::printCount();

r1.introduce();
r2.introduce();

// Can also call via instance (though class-scope syntax is preferred)
r1.printCount();

## Why No `this` in Static Methods

The `this` pointer is implicitly passed to every **non-static** member function — it points to the specific instance the method is called on.

A static method has no instance. There is nothing to pass `this` for. Therefore:

```
error: invalid use of member 'Robot::_name' in static member function
```

If you need per-object data inside a method, it must be a **non-static** method. If a method only needs class-level (shared) data, make it static.

## Static vs Instance Members

Static data is **shared** — changing it through one object changes it for all. Instance data is **per-object**.

In [ ]:
#include <iostream>

class Config {
public:
    int         instanceValue;   // per-object
    static int  sharedValue;     // shared by all

    Config(int iv) : instanceValue(iv) {}
};

int Config::sharedValue = 0;

Config a(10);
Config b(20);

std::cout << "a.instanceValue = " << a.instanceValue << std::endl;
std::cout << "b.instanceValue = " << b.instanceValue << std::endl;
std::cout << "sharedValue     = " << Config::sharedValue << std::endl;

// Change shared value through 'a'
a.sharedValue = 42;

// Visible through 'b' as well
std::cout << "\nAfter a.sharedValue = 42:" << std::endl;
std::cout << "a.sharedValue = " << a.sharedValue << std::endl;
std::cout << "b.sharedValue = " << b.sharedValue << std::endl;   // also 42

**Exercise 2:** Create a class `Temperature` with:
- A `static double conversionOffset` (default 273.15 for Kelvin).
- A static setter `setOffset(double offset)`.
- An instance member `double celsius`.
- A method `double toKelvin() const` that returns `celsius + conversionOffset`.

Create two `Temperature` objects, print their Kelvin values, change the offset, and print again.

In [ ]:
// Your code here

## The Singleton Pattern

A **Singleton** ensures only one instance of a class can ever exist. It is a classic use of static members:
- Private constructor — nobody can call `new Singleton()` from outside.
- Static pointer holding the single instance.
- Static `getInstance()` that creates the instance on first call and returns it every time.

> Note: Singletons are convenient but can make testing harder and hide dependencies. Use them with awareness.

In [ ]:
#include <iostream>

class AppConfig {
public:
    static AppConfig *getInstance() {
        if (_instance == NULL)
            _instance = new AppConfig();
        return _instance;
    }

    // In a real app you would also provide a destroy() method
    static void destroy() {
        delete _instance;
        _instance = NULL;
    }

    void setMaxUsers(int n) { _maxUsers = n; }
    int  getMaxUsers() const { return _maxUsers; }

private:
    AppConfig() : _maxUsers(10) {
        std::cout << "[AppConfig] created" << std::endl;
    }

    ~AppConfig() {
        std::cout << "[AppConfig] destroyed" << std::endl;
    }

    int               _maxUsers;
    static AppConfig *_instance;
};

AppConfig *AppConfig::_instance = NULL;

AppConfig *cfg1 = AppConfig::getInstance();   // creates instance
AppConfig *cfg2 = AppConfig::getInstance();   // returns same instance

std::cout << "Same pointer? " << (cfg1 == cfg2 ? "YES" : "NO") << std::endl;

cfg1->setMaxUsers(42);
std::cout << "maxUsers via cfg2: " << cfg2->getMaxUsers() << std::endl;

AppConfig::destroy();

## `const static` Members

For integral types (`int`, `char`, `bool`, etc.), C++98 allows a `const static` member to be **initialised inside the class declaration**:

```cpp
class Limits {
public:
    static const int MAX_SIZE = 100;   // OK in C++98 for integral types
};
```

This makes compile-time constants that belong to the class scope — a cleaner alternative to preprocessor `#define`.

In [ ]:
#include <iostream>

class NetworkConfig {
public:
    static const int    MAX_CONNECTIONS = 128;
    static const int    DEFAULT_PORT    = 4242;
    static const char   PROTOCOL        = 'T';  // T for TCP

    static void printDefaults() {
        std::cout << "Max connections: " << MAX_CONNECTIONS << std::endl;
        std::cout << "Default port:    " << DEFAULT_PORT    << std::endl;
        std::cout << "Protocol:        " << PROTOCOL        << std::endl;
    }
};

NetworkConfig::printDefaults();
std::cout << "Access without object: port=" << NetworkConfig::DEFAULT_PORT << std::endl;

## Practical Use Cases

- **Counting instances**: track how many objects of a type currently exist.
- **Shared configuration**: one config object used by all instances.
- **Utility functions**: functions that belong logically to the class but need no per-object state (e.g., `Math::sqrt`, `StringUtils::toUpper`).
- **Factory methods**: static `create()` method that decides which subclass to instantiate.
- **Singleton**: exactly one instance of a resource (logger, config, database connection).

In [ ]:
#include <iostream>
#include <string>

// Utility class: all methods static, no data members, never instantiated
class MathUtils {
public:
    static int    clamp(int value, int lo, int hi) {
        if (value < lo) return lo;
        if (value > hi) return hi;
        return value;
    }

    static int    abs_val(int x)   { return x < 0 ? -x : x; }

    static int    max_val(int a, int b) { return a > b ? a : b; }

private:
    MathUtils();   // prevent instantiation
};

std::cout << "clamp(150, 0, 100) = " << MathUtils::clamp(150, 0, 100) << std::endl;
std::cout << "abs_val(-42)       = " << MathUtils::abs_val(-42)       << std::endl;
std::cout << "max_val(7, 13)     = " << MathUtils::max_val(7, 13)     << std::endl;

## Final Exercise

**Exercise 3:** Implement a `class Logger` with the following:
- A `static int logLevel` (1 = debug, 2 = info, 3 = error). Default: 1.
- A static method `setLogLevel(int level)`.
- A static method `log(int level, const std::string &msg)` that prints the message only if `level >= logLevel`. Format: `[DEBUG]`, `[INFO ]`, or `[ERROR]` prefix depending on level.
- A static `int callCount` that tracks how many times `log()` was called (regardless of whether anything was printed).
- A static `getCallCount()` method.

Test by calling `log` with various levels, changing the log level, and verifying the call count.

In [ ]:
// Your code here

## Modern C++ (C++11 and Beyond)

### `static_assert` — Compile-time assertions

```cpp
static_assert(sizeof(int) == 4, "This code requires 32-bit int");
```

The error is raised at **compile time**, not at runtime. Useful for checking platform assumptions.

### `constexpr static` members

C++11 allows `constexpr static` for any literal type (not just integrals):

```cpp
static constexpr double PI = 3.14159265358979;
```

### Thread-safe static locals (C++11 guarantee)

In C++11, the initialisation of a `static` local variable is guaranteed to be **thread-safe** — only one thread will execute the initialiser. This makes the Meyers Singleton safe:

```cpp
static AppConfig &getInstance() {
    static AppConfig instance;   // C++11: thread-safe, no pointer needed
    return instance;
}
```

In [ ]:
#include <iostream>

// static_assert: checked at compile time
static_assert(sizeof(int) >= 2, "int must be at least 16 bits");
static_assert(sizeof(char) == 1, "char must be exactly 1 byte");

class Circle {
public:
    // constexpr static: usable in constant expressions
    static constexpr double PI = 3.14159265358979;

    Circle(double radius) : _radius(radius) {}

    double area()        const { return PI * _radius * _radius; }
    double circumference() const { return 2.0 * PI * _radius; }

private:
    double _radius;
};

Circle c(5.0);
std::cout << "Area:          " << c.area()          << std::endl;
std::cout << "Circumference: " << c.circumference() << std::endl;
std::cout << "PI constant:   " << Circle::PI        << std::endl;

// Meyers Singleton (C++11 thread-safe)
class Registry {
public:
    static Registry &getInstance() {
        static Registry instance;   // constructed once, thread-safe in C++11
        return instance;
    }

    void set(int v) { _value = v; }
    int  get() const { return _value; }

private:
    Registry() : _value(0) {}
    int _value;
};

Registry::getInstance().set(42);
std::cout << "Registry value: " << Registry::getInstance().get() << std::endl;